# DisorderNet GPU: ESM-2 650M + LoRA on Colab Pro

**Goal:** AUC-ROC > 0.88 on DisProt (5-fold protein-grouped CV)

| Component | Detail |
|---|---|
| Hardware | Colab Pro GPU — A100 (40/80 GB), L4, or T4 (auto-tuned batch size) |
| Backbone | ESM-2 650M (33 layers, 1280-dim) |
| Fine-tuning | LoRA rank 8 on Q/V projections (last 8 layers) |
| Head | 1D CNN + residual skip |
| Data | DisProt (disorder-term filtered annotations) |

### Before you run
1. **Runtime → Change runtime type → GPU** (A100 or L4 recommended)
2. **Runtime → Change runtime type → High-RAM** if available
3. Run all cells top-to-bottom (~12–18 h for full 5-fold CV on A100)

Live progress: tqdm bars per epoch + per-epoch metric lines. Checkpoints saved under `checkpoints/`.

In [ ]:
# ── CELL 1 │ Clone repo (Colab only downloads the notebook by default) ──
import os, sys, subprocess

REPO_URL = "https://github.com/Tommaso-R-Marena/DisorderNet.git"
REPO_DIR = "DisorderNet"

# Already inside the cloned repo (e.g. opened from GitHub file browser)
if os.path.isfile("colab/disordernet_gpu.py"):
    if "." not in sys.path:
        sys.path.insert(0, ".")
    print("✅ Running from repository root.")
elif os.path.isdir(REPO_DIR):
    sys.path.insert(0, REPO_DIR)
    print(f"✅ Repository already at '{REPO_DIR}/'")
else:
    print("Cloning DisorderNet repository...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    sys.path.insert(0, REPO_DIR)
    print("✅ Repository cloned.")

In [ ]:
# ── CELL 2 │ Environment setup ─────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")

from colab.disordernet_gpu import (
    TrainConfig,
    install_dependencies,
    setup_environment,
    fetch_disprot,
    process_disprot,
    print_dataset_summary,
    load_esm_model,
    DisorderNetGPU,
    DisProtDataset,
    disprot_collate,
    run_cross_validation,
    _in_colab,
)

install_dependencies(quiet=True)

# ── Hyperparameters ─────────────────────────────────────────────────────
# QUALITY_PROFILE: "balanced" (default v2) or "max" (rank 32, 25 epochs, 48-dim physico)
QUALITY_PROFILE = "max"

cfg = TrainConfig.from_profile(
    QUALITY_PROFILE,
    batch_size=4,          # auto-overridden for your GPU VRAM
    accum_steps=4,
    n_folds=5,
    use_gradient_checkpointing=True,
    deterministic=False,   # False = TF32 + cudnn.benchmark (faster)
)

cfg = setup_environment(cfg)
DEVICE = cfg.device
print(f"\nColab: {_in_colab()}  |  Seed: {cfg.seed}  |  Profile: {QUALITY_PROFILE}")
print(f"  GPU: {cfg.gpu_name}  |  VRAM: {cfg.vram_gb:.1f} GB  |  AMP: {cfg.amp_dtype}")
print(f"  Effective batch  : {cfg.effective_batch()}  |  Folds: {cfg.n_folds}")

In [ ]:
# ── CELL 3 │ Optional: mount Google Drive for checkpoint persistence ───
import os

MOUNT_DRIVE = False   # ← set True to save checkpoints across Colab sessions
DRIVE_SUBDIR = "DisorderNet"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_ckpt = f"/content/drive/MyDrive/{DRIVE_SUBDIR}/checkpoints"
    os.makedirs(drive_ckpt, exist_ok=True)
    cfg.checkpoint_dir = drive_ckpt
    print(f"Checkpoints → {cfg.checkpoint_dir}")
else:
    print("Google Drive not mounted. Checkpoints saved locally in ./checkpoints/")

In [ ]:
# ── CELL 4 │ Download & process DisProt ────────────────────────────────
import json

entries = fetch_disprot(cache_path=cfg.data_cache)
proteins, skipped = process_disprot(entries, cfg)
total_res, total_dis, frac_dis = print_dataset_summary(proteins, skipped)

assert len(proteins) >= 500, f"Expected ≥500 proteins, got {len(proteins)}"
print("\n✅ Data validation passed.")
print("   Disorder labels: disorder-related term_names only.")
print("   Functional regions preserved for biological-utility evaluation.")

In [ ]:
# ── CELL 5 │ Load ESM-2 650M ───────────────────────────────────────────
import torch

model_esm, alphabet, batch_converter = load_esm_model(DEVICE)

mem_gb = torch.cuda.memory_allocated() / 1e9
print(f"  VRAM used: {mem_gb:.1f} GB / {cfg.vram_gb:.1f} GB")

# Quick forward-pass sanity check
_, _, test_tokens = batch_converter([("test", "ACDEFGHIKLMNPQRSTVWY" * 3)])
with torch.inference_mode():
    out = model_esm(test_tokens.to(DEVICE), repr_layers=[33], return_contacts=False)
print(f"  Embedding shape: {out['representations'][33].shape}")
del test_tokens, out
torch.cuda.empty_cache()
print("✅ ESM-2 verified.")

In [ ]:
# ── CELL 6 │ Build model + smoke-test DataLoader ───────────────────────
import torch
from torch.utils.data import DataLoader
from colab.disordernet_gpu import print_training_config_summary

print("Building DisorderNetGPU...")
print_training_config_summary(cfg)
model = DisorderNetGPU(model_esm, cfg).to(DEVICE)
print(f"  VRAM after build: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

print("\nSmoke-test DataLoader (8 proteins)...")
test_ds = DisProtDataset(
    proteins[:8], batch_converter, cfg=cfg,
    plddt_by_id=globals().get("plddt_by_protein"),
)
test_dl = DataLoader(test_ds, batch_size=2, collate_fn=disprot_collate, num_workers=0)
tok, lab, msk, aa_idx, sample_wt, ids = next(iter(test_dl))
print(f"  Batch shapes     : tokens={tuple(tok.shape)}  labels={tuple(lab.shape)}")
print(f"  Protein IDs      : {ids}")
print(f"  Disorder frac    : {lab[msk].mean():.3f}")
print(f"  Sample wt range  : {sample_wt[msk].min():.2f} – {sample_wt[msk].max():.2f}")

print("  Forward pass...")
with torch.inference_mode():
    logits = model(tok.to(DEVICE), aa_idx=aa_idx.to(DEVICE))
    probs = torch.sigmoid(logits[msk.to(DEVICE)])
print(f"  Logits / probs   : {tuple(logits.shape)}  mean_prob={probs.mean():.3f}")

del test_ds, test_dl, model, tok, lab, msk, aa_idx, sample_wt, logits, probs
torch.cuda.empty_cache()
print("\n✅ Model + DataLoader + forward pass verified.")

In [ ]:
# ── CELL 6b │ Pre-fetch AF pLDDT (hallucination hard-negative weighting) ──
# Fetches AlphaFold pLDDT before CV so training can upweight disordered residues
# with high pLDDT (AF "hallucinated order"). Safe to skip if cache already warm.

from colab.disordernet_gpu import build_plddt_cache_for_training

PREFETCH_AF_PLDDT = True   # set False to disable hallucination weighting prefetch
AF_CACHE_DIR = "af_plddt_cache"

plddt_by_protein = {}
if PREFETCH_AF_PLDDT and cfg.use_hallucination_weighting:
    print(f"Fetching AF pLDDT for {len(proteins)} proteins (UniProt required)...")
    plddt_by_protein = build_plddt_cache_for_training(
        proteins,
        cache_dir=AF_CACHE_DIR,
        sleep_s=0.05,
    )
    n_uni = sum(1 for p in proteins if p.get("uniprot_acc"))
    print(f"✅ pLDDT cache: {len(plddt_by_protein)}/{len(proteins)} proteins "
          f"({n_uni} with UniProt acc)")
else:
    print("Skipping AF pLDDT prefetch (hallucination weighting uses cache-only hits)")

In [ ]:
# ── CELL 7 │ 5-Fold Cross-Validation ───────────────────────────────────
# Resume support: if Colab disconnected, set RESUME_FROM_FOLD to the next fold (0-based)
RESUME_FROM_FOLD = 0

from IPython.display import display, HTML, clear_output
import pandas as pd
from datetime import datetime
from colab.disordernet_gpu import print_training_config_summary

print(f"\n{'═' * 60}")
print(f" 5-FOLD CV START  {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")
print(f"{'═' * 60}")
print_training_config_summary(cfg, proteins)
est_h = cfg.n_folds * cfg.num_epochs * 0.15  # rough hours/fold-epoch on A100
print(f"  Rough ETA        : {est_h:.0f}–{est_h * 1.5:.0f} h on A100 (varies by length)")
print(f"  Resume from fold : {RESUME_FROM_FOLD}")

live_rows = []
fold_progress = []

def _on_epoch(row):
    live_rows.append(row)
    clear_output(wait=True)
    display(HTML("<b>Training progress (last 12 epochs)</b>"))
    cols = ["epoch", "train_loss", "val_auc", "val_ap", "val_segment_f1", "val_f1"]
    df = pd.DataFrame(live_rows[-12:])
    show = [c for c in cols if c in df.columns]
    display(df[show] if show else df)
    if fold_progress:
        display(HTML("<b>Completed folds</b>"))
        display(pd.DataFrame(fold_progress))

def _on_fold_done(result):
    fold_progress.append({
        "fold": result["fold"],
        "best_auc": round(result["best_auc"], 4),
        "best_ap": round(result["best_ap"], 4),
        "hours": round(result["total_time"] / 3600, 2),
    })
    print(f"\n✓ Fold {result['fold']} done — AUC={result['best_auc']:.4f} "
          f"AP={result['best_ap']:.4f} ({result['total_time']/3600:.2f} h)")

fold_results, cv_summary = run_cross_validation(
    proteins=proteins,
    esm_backbone=model_esm,
    batch_converter=batch_converter,
    cfg=cfg,
    resume_from_fold=RESUME_FROM_FOLD,
    plddt_by_id=plddt_by_protein if plddt_by_protein else None,
    prefetch_af_plddt=False,  # already prefetched in Cell 6b
    on_epoch_end=_on_epoch,
    on_fold_complete=_on_fold_done,
)

clear_output(wait=True)
print(f"\n{'═' * 60}")
print(" CV COMPLETE")
print(f"{'═' * 60}")
pooled_auc = cv_summary["pooled_auc"]
pooled_ap  = cv_summary["pooled_ap"]
fold_aucs  = cv_summary["fold_aucs"]
total_cv_h = cv_summary["total_cv_hours"]
print(f"  Pooled AUC/AP    : {pooled_auc:.4f} / {pooled_ap:.4f}")
print(f"  Fold AUCs        : {[round(a, 4) for a in fold_aucs]}")
print(f"  Mean ± std       : {cv_summary['mean_auc']:.4f} ± {cv_summary['std_auc']:.4f}")
print(f"  Total time       : {total_cv_h:.2f} h")

all_probs  = __import__("numpy").concatenate([r["val_probs"] for r in fold_results])
all_labels = __import__("numpy").concatenate([r["val_labels"] for r in fold_results])

with open("cv_results.json", "w") as f:
    json.dump(cv_summary, f, indent=2)
print("\nSaved cv_results.json")

if pooled_auc >= 0.88:
    print(f"\n🎉 TARGET AUC ≥ 0.88 ACHIEVED! ({pooled_auc:.4f})")
elif pooled_auc >= 0.83:
    print(f"\n✅ Beats DisorderNet v6 baseline (0.831). AUC = {pooled_auc:.4f}")

In [ ]:
# ── CELL 7b │ GPU + v6 CPU ensemble ───────────────────────────────────────
# Blends GPU OOF predictions with lightweight v6-style CPU model (features_fast + HGB).
# Blend weight w is tuned on OOF residues to lift pooled AUC toward v6 (0.831).

from colab.ensemble_v6 import (
    apply_gpu_v6_ensemble,
    print_ensemble_report,
    save_ensemble_report,
)

APPLY_GPU_V6_ENSEMBLE = True

if APPLY_GPU_V6_ENSEMBLE:
    ensemble_report, fold_results, v6_probs_by_protein = apply_gpu_v6_ensemble(
        proteins=proteins,
        fold_results=fold_results,
        n_folds=cfg.n_folds,
        weight=None,
        v6_cache_path="v6_oof_probs_cache.json",
        run_v6_if_missing=True,
        seed=cfg.seed,
    )
    print_ensemble_report(ensemble_report)
    save_ensemble_report(ensemble_report, "gpu_v6_ensemble_report.json")

    pooled_auc = ensemble_report["after"]["pooled"]["auc"]
    pooled_ap = ensemble_report["after"]["pooled"]["ap"]
    all_probs = __import__("numpy").concatenate([r["val_probs"] for r in fold_results])
    all_labels = __import__("numpy").concatenate([r["val_labels"] for r in fold_results])
    cv_summary["pooled_auc"] = pooled_auc
    cv_summary["pooled_ap"] = pooled_ap
    cv_summary["gpu_v6_ensemble"] = ensemble_report
    print("✅ GPU+v6 ensemble applied; downstream metrics will use blended predictions")
else:
    ensemble_report = {"skipped": True, "reason": "APPLY_GPU_V6_ENSEMBLE=False"}
    v6_probs_by_protein = {}

In [ ]:
# ── CELL 8 │ CAID metrics + matched benchmark tables ───────────────────
from colab.colab_figures import optimal_threshold
from colab.benchmark_tables import print_matched_benchmark_report
from colab.caid_reporting import run_full_caid_report, print_caid_report, save_caid_report
from sklearn.metrics import f1_score, matthews_corrcoef

our_auc = pooled_auc
our_ap  = pooled_ap
opt_thresh, _ = optimal_threshold(all_labels, all_probs)
preds_opt = (all_probs >= opt_thresh).astype(int)
our_f1  = f1_score(all_labels.astype(int), preds_opt)
our_mcc = matthews_corrcoef(all_labels.astype(int), preds_opt)

caid_report = run_full_caid_report(
    proteins=proteins,
    fold_results=fold_results,
    threshold=opt_thresh,
    n_folds=cfg.n_folds,
)
print_caid_report(caid_report)
save_caid_report(caid_report, "caid_evaluation_report.json")

f1_max = caid_report["stratified"]["pooled"].get("f1_max")
benchmark_report = print_matched_benchmark_report(
    gpu_auc=our_auc,
    gpu_ap=our_ap,
    gpu_f1_max=f1_max,
    gpu_mcc=our_mcc,
)

print(f"\n  F1@opt={our_f1:.4f}  MCC@opt={our_mcc:.4f}  F1_max={f1_max:.4f}  opt_thresh={opt_thresh:.3f}")
print("  ✅ Saved caid_evaluation_report.json")

In [ ]:
# ── CELL 9 │ Biological utility report (Phase 1) ─────────────────────────
from colab.biological_utility import (
    run_biological_utility_report,
    print_biological_utility_report,
    save_biological_utility_report,
)
from colab.colab_figures import generate_biological_utility_figure

bio_report = run_biological_utility_report(
    proteins=proteins,
    fold_results=fold_results,
    threshold=opt_thresh,
    n_folds=cfg.n_folds,
)
print_biological_utility_report(bio_report)
save_biological_utility_report(bio_report, "biological_utility_report.json")
generate_biological_utility_figure(bio_report)
print("\n✅ Saved biological_utility_report.json and fig5_biological_utility.{pdf,png}")

In [ ]:
# ── CELL 10 │ Phase 2: AlphaFold pLDDT hallucination rescue ────────────
from colab.af_hallucination import (
    fetch_and_run_af_rescue_report,
    print_af_rescue_report,
    save_af_rescue_report,
)
from colab.colab_figures import generate_af_rescue_figure

HIGH_PLDDT = 70.0   # AF "confident" threshold; disordered + high pLDDT = hallucination
AF_CACHE_DIR = "af_plddt_cache"

af_report, plddt_by_protein = fetch_and_run_af_rescue_report(
    proteins=proteins,
    fold_results=fold_results,
    threshold=opt_thresh,
    high_plddt_threshold=HIGH_PLDDT,
    n_folds=cfg.n_folds,
    cache_dir=AF_CACHE_DIR,
    sleep_s=0.05,
)
print_af_rescue_report(af_report)
save_af_rescue_report(af_report, "af_rescue_report.json")
generate_af_rescue_figure(af_report)
print("\n✅ Saved af_rescue_report.json and fig6_af_rescue.{pdf,png}")

In [ ]:
# ── CELL 10b │ AF–DisorderNet fusion (inference boost) ───────────────────
# Blends CV predictions with inverse-pLDDT where AlphaFold coverage exists.
# Prior run: α=0.50 lifted AF-subset AUC 0.816 → 0.831. Updates all downstream metrics.

from colab.inference_fusion import (
    apply_plddt_fusion_to_cv,
    print_fusion_report,
    save_fusion_report,
)
from colab.colab_figures import optimal_threshold
from colab.caid_reporting import run_full_caid_report, print_caid_report, save_caid_report
from colab.benchmark_tables import print_matched_benchmark_report
from colab.biological_utility import (
    run_biological_utility_report,
    print_biological_utility_report,
    save_biological_utility_report,
)
from sklearn.metrics import f1_score, matthews_corrcoef

APPLY_AF_FUSION = True   # set False to keep raw DisorderNet-only predictions

import copy
fold_results_gpu = copy.deepcopy(fold_results)  # kept for Cell 11b fusion comparison

if APPLY_AF_FUSION and plddt_by_protein:
    fusion_report, fold_results = apply_plddt_fusion_to_cv(
        proteins=proteins,
        fold_results=fold_results,
        plddt_by_protein=plddt_by_protein,
        n_folds=cfg.n_folds,
        alpha=None,
    )
    print_fusion_report(fusion_report)
    save_fusion_report(fusion_report, "inference_fusion_report.json")

    all_probs = __import__("numpy").concatenate([r["val_probs"] for r in fold_results])
    all_labels = __import__("numpy").concatenate([r["val_labels"] for r in fold_results])
    our_auc = fusion_report["after"]["pooled"]["auc"]
    our_ap = fusion_report["after"]["pooled"]["ap"]

    opt_thresh, _ = optimal_threshold(all_labels, all_probs)
    preds_opt = (all_probs >= opt_thresh).astype(int)
    our_f1 = f1_score(all_labels.astype(int), preds_opt)
    our_mcc = matthews_corrcoef(all_labels.astype(int), preds_opt)

    caid_report = run_full_caid_report(
        proteins=proteins, fold_results=fold_results,
        threshold=opt_thresh, n_folds=cfg.n_folds,
    )
    print_caid_report(caid_report)
    save_caid_report(caid_report, "caid_evaluation_report.json")

    bio_report = run_biological_utility_report(
        proteins=proteins, fold_results=fold_results,
        threshold=opt_thresh, n_folds=cfg.n_folds,
    )
    print_biological_utility_report(bio_report)
    save_biological_utility_report(bio_report, "biological_utility_report.json")

    f1_max = caid_report["stratified"]["pooled"].get("f1_max")
    benchmark_report = print_matched_benchmark_report(
        gpu_auc=our_auc, gpu_ap=our_ap, gpu_f1_max=f1_max, gpu_mcc=our_mcc,
    )

    print(f"\n✅ Fused metrics: AUC={our_auc:.4f}  AP={our_ap:.4f}  F1_max={f1_max:.4f}")
    print("   Saved inference_fusion_report.json; refreshed caid + biological utility reports")
else:
    fusion_report = {"skipped": True, "reason": "APPLY_AF_FUSION=False or no pLDDT"}
    print("Skipping AF fusion — run Cell 10 first or set APPLY_AF_FUSION=True")
    fold_results_gpu = copy.deepcopy(fold_results)

In [ ]:
# ── CELL 11 │ Phase 2b: AlphaFold 3 — auto Drive + GitHub pipeline ─────
# Drive path (from your screenshot):
#   MyDrive/DisorderNet/af3/af3.bin
#   MyDrive/DisorderNet/af3/outputs/
#
# Modes:
#   "auto"   — clone AF3 GitHub, use Drive weights, run ALL missing proteins, ingest
#   "ingest" — load precomputed outputs from Drive only
#   "run"    — run a capped subset (AF3_MAX_RUN)
#   "off"    — skip AF3

from colab.af3_colab import (
    DEFAULT_DRIVE_ROOT,
    print_af3_setup_instructions,
    run_af3_auto_pipeline,
    run_af3_subset_on_colab,
    select_proteins_for_af3,
    setup_af3_for_colab,
)
from colab.af_hallucination import (
    fetch_and_run_af3_rescue_report,
    run_af2_af3_comparison_report,
    print_af2_af3_comparison,
    print_af_rescue_report,
    save_af2_af3_comparison,
    save_af_rescue_report,
)
from colab.colab_figures import generate_af2_af3_comparison_figure

AF3_MODE = "auto"       # "auto" | "ingest" | "run" | "off"
AF3_DRIVE_ROOT = DEFAULT_DRIVE_ROOT  # /content/drive/MyDrive/DisorderNet/af3
AF3_MAX_RUN = None      # None = all missing proteins; set e.g. 25 to cap per session
AF3_TIMEOUT_S = 3600    # per-protein timeout (seconds)
AF3_MSA_FREE = True     # True = no 630 GB DBs (empty MSA); False needs public_databases/
AF3_CACHE_DIR = "af3_plddt_cache"

af3_report = {"insufficient_data": True, "skipped": True}
af3_comparison = {"insufficient_data": True}
plddt_af3_by_protein = {}
af3_batch_result = {}
breakthrough_summary = {"insufficient_data": True}

af3_cfg = setup_af3_for_colab(
    mode=AF3_MODE,
    drive_root=AF3_DRIVE_ROOT,
    mount_drive=True,
    clone_repo=True,
)
print_af3_setup_instructions(af3_cfg["paths"])
print(f"  Docker available : {af3_cfg.get('docker_ok')}")
print(f"  AF3 repo cloned  : {af3_cfg.get('repo_ok')}")
print(f"  Genetic DBs      : {af3_cfg.get('databases_ok')}")

done, pending = select_proteins_for_af3(proteins, af3_cfg["paths"]["output_dir"])
print(f"  AF3 coverage     : {len(done)} done, {len(pending)} pending (UniProt proteins)")

if AF3_MODE == "off":
    print("AF3 disabled (AF3_MODE='off'). Using AF2 results from Cell 10 only.")
elif AF3_MODE == "auto":
    if not af3_cfg.get("ready"):
        print("⚠️ AF3 auto prerequisites missing:", af3_cfg.get("weights_message", ""))
    elif pending:
        af3_batch_result = run_af3_auto_pipeline(
            proteins=proteins,
            paths=af3_cfg["paths"],
            max_proteins=AF3_MAX_RUN,
            timeout_s=AF3_TIMEOUT_S,
            msa_free=AF3_MSA_FREE,
        )
        print("AF3 auto batch:", af3_batch_result)
        cov = af3_batch_result.get("coverage_after", {})
        print(f"  Coverage after run: {cov.get('n_done', 0)}/{cov.get('n_eligible', 0)} "
              f"({100 * cov.get('fraction', 0):.1f}%)")
    else:
        print("All eligible proteins already have AF3 outputs — ingesting only.")
elif AF3_MODE == "run":
    if not af3_cfg.get("ready"):
        print("⚠️ AF3 run prerequisites missing:", af3_cfg.get("weights_message", ""))
    else:
        cap = AF3_MAX_RUN if AF3_MAX_RUN is not None else 25
        af3_batch_result = run_af3_subset_on_colab(
            proteins=proteins,
            paths=af3_cfg["paths"],
            max_proteins=cap,
            timeout_s=AF3_TIMEOUT_S,
            msa_free=AF3_MSA_FREE,
        )
        print("AF3 subset run:", af3_batch_result)

if AF3_MODE in ("auto", "ingest", "run") and af3_cfg["paths"]["output_dir"]:
    af3_report, plddt_af3_by_protein = fetch_and_run_af3_rescue_report(
        proteins=proteins,
        fold_results=fold_results,
        af3_output_root=af3_cfg["paths"]["output_dir"],
        threshold=opt_thresh,
        high_plddt_threshold=HIGH_PLDDT,
        n_folds=cfg.n_folds,
        cache_dir=AF3_CACHE_DIR,
    )
    af3_report["skipped"] = False
    af3_report["batch_result"] = af3_batch_result
    print_af_rescue_report(af3_report)
    save_af_rescue_report(af3_report, "af3_rescue_report.json")

    af3_comparison = run_af2_af3_comparison_report(af_report, af3_report)
    print_af2_af3_comparison(af3_comparison)
    save_af2_af3_comparison(af3_comparison, "af2_af3_comparison.json")
    generate_af2_af3_comparison_figure(af3_comparison, af2_report=af_report, af3_report=af3_report)
    print("\n✅ Saved af3_rescue_report.json, af2_af3_comparison.json, fig7_af2_af3_comparison.{pdf,png}")
else:
    print("No AF3 outputs — check Drive path or set AF3_MODE='auto'.")

In [ ]:
# ── CELL 11b │ AF2 vs AF3 breakthrough summary (3 headline comparisons) ─
# 1. AF3 vs AF2 hallucination rate (pooled + same-protein overlap)
# 2. DisorderNet rescue stronger against AF3 than AF2?
# 3. Fusion lift vs pLDDT baseline: larger on AF3 than AF2?

from colab.af_hallucination import (
    run_af2_af3_breakthrough_summary,
    print_af2_af3_breakthrough_summary,
    save_af2_af3_breakthrough_summary,
)

breakthrough_summary = {"insufficient_data": True}

if not af3_report.get("skipped") and not af3_report.get("insufficient_data"):
    _gpu_folds = fold_results_gpu if "fold_results_gpu" in dir() else fold_results
    _fusion_alpha = None
    if "fusion_report" in dir() and fusion_report.get("fusion_alpha") is not None:
        _fusion_alpha = fusion_report["fusion_alpha"]

    breakthrough_summary = run_af2_af3_breakthrough_summary(
        proteins=proteins,
        fold_results_gpu=_gpu_folds,
        af2_report=af_report,
        af3_report=af3_report,
        af2_af3_comparison=af3_comparison,
        plddt_af2_by_protein=plddt_by_protein,
        plddt_af3_by_protein=plddt_af3_by_protein,
        fusion_alpha=_fusion_alpha,
        fusion_report=fusion_report if "fusion_report" in dir() else None,
        threshold=opt_thresh,
        high_plddt_threshold=HIGH_PLDDT,
        n_folds=cfg.n_folds,
    )
    print_af2_af3_breakthrough_summary(breakthrough_summary)
    save_af2_af3_breakthrough_summary(breakthrough_summary, "af2_af3_breakthrough_summary.json")
    print("\n✅ Saved af2_af3_breakthrough_summary.json")
else:
    print("Skipping breakthrough summary — run Cell 11 with AF3 outputs first.")

In [ ]:
# ── CELL 11c │ AF3-enhanced fusion + refresh all downstream metrics ─────
# Prefer AF3 pLDDT where available, fall back to AF2. Re-runs CAID, bio utility,
# and benchmark tables with updated fold_results.

from colab.inference_fusion import (
    apply_combined_plddt_fusion_to_cv,
    print_fusion_report,
    save_fusion_report,
)
from colab.downstream_refresh import refresh_downstream_metrics

APPLY_AF3_FUSION = True
af3_fusion_report = {"skipped": True}

if APPLY_AF3_FUSION and plddt_af3_by_protein and plddt_by_protein:
    _base_folds = fold_results_gpu if "fold_results_gpu" in dir() else fold_results
    af3_fusion_report, fold_results = apply_combined_plddt_fusion_to_cv(
        proteins=proteins,
        fold_results=_base_folds,
        plddt_af2=plddt_by_protein,
        plddt_af3=plddt_af3_by_protein,
        prefer="af3",
        n_folds=cfg.n_folds,
        alpha=None,
    )
    af3_fusion_report["skipped"] = False
    print_fusion_report(af3_fusion_report)
    save_fusion_report(af3_fusion_report, "af3_combined_fusion_report.json")

    refreshed = refresh_downstream_metrics(
        proteins=proteins,
        fold_results=fold_results,
        n_folds=cfg.n_folds,
    )
    our_auc = refreshed["our_auc"]
    our_ap = refreshed["our_ap"]
    our_f1 = refreshed["our_f1"]
    our_mcc = refreshed["our_mcc"]
    opt_thresh = refreshed["opt_threshold"]
    caid_report = refreshed["caid_report"]
    bio_report = refreshed["bio_report"]
    benchmark_report = refreshed["benchmark_report"]
    all_probs = refreshed["all_probs"]
    all_labels = refreshed["all_labels"]
    print(f"\n✅ AF3-enhanced fusion applied. Seg F1={refreshed.get('segment_f1', 'N/A')}")
else:
    print("Skipping AF3 fusion — need both AF2 (Cell 10) and AF3 (Cell 11) pLDDT.")

In [ ]:
# ── CELL 12 │ Phase 3: Structure calibration & integrated synthesis ────
from colab.phase3_synthesis import (
    run_structure_calibration_report,
    run_phase3_integrated_report,
    print_phase3_report,
    save_phase3_report,
)
from colab.statistical_validation import (
    run_full_statistical_validation,
    print_statistical_validation,
    save_statistical_validation,
)
from colab.colab_figures import generate_phase3_figure

calibration_report = run_structure_calibration_report(
    proteins=proteins,
    fold_results=fold_results,
    plddt_by_protein=plddt_by_protein,
    threshold=opt_thresh,
    high_plddt_threshold=HIGH_PLDDT,
    n_folds=cfg.n_folds,
    n_boot=500,
    af_source="AlphaFold DB (AF2)",
)

stat_validation = run_full_statistical_validation(
    proteins=proteins,
    fold_results=fold_results,
    plddt_by_protein=plddt_by_protein,
    n_folds=cfg.n_folds,
)
print_statistical_validation(stat_validation)
save_statistical_validation(stat_validation, "statistical_validation_report.json")

cv_pooled = {
    "auc": float(our_auc),
    "ap": float(our_ap),
    "f1": float(our_f1),
    "mcc": float(our_mcc),
    "opt_threshold": float(opt_thresh),
    "f1_max": caid_report["stratified"]["pooled"].get("f1_max"),
}

af3_for_phase3 = af3_report if not af3_report.get("skipped") else None
comparison_for_phase3 = (
    af3_comparison if not af3_comparison.get("insufficient_data") else None
)

phase3_report = run_phase3_integrated_report(
    cv_pooled=cv_pooled,
    bio_report=bio_report,
    af_report=af_report,
    calibration_report=calibration_report,
    af3_report=af3_for_phase3,
    af2_af3_comparison=comparison_for_phase3,
    caid_report=caid_report,
    statistical_validation=stat_validation,
    benchmark_report=benchmark_report,
)
print_phase3_report(phase3_report)
save_phase3_report(phase3_report, "phase3_integrated_report.json")
generate_phase3_figure(phase3_report)
print("\n✅ Saved phase3_integrated_report.json, statistical_validation_report.json, fig8_phase3_synthesis.{pdf,png}")

In [ ]:
# ── CELL 13 │ Figures ──────────────────────────────────────────────────
from colab.colab_figures import generate_all_figures

fig_metrics = generate_all_figures(
    fold_results, all_labels, all_probs, our_auc, our_ap
)
opt_thresh = fig_metrics["opt_thresh"]
our_f1     = fig_metrics["f1"]
our_mcc    = fig_metrics["mcc"]
print("✅ All figures saved.")

In [ ]:
# ── CELL 14 │ Save results & best checkpoint ───────────────────────────
import shutil
import numpy as np
from datetime import datetime

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
best_fold_idx = int(np.argmax([r["best_auc"] for r in fold_results]))
best_fold     = fold_results[best_fold_idx]
best_ckpt_dst = f"disordernet_gpu_best_{RUN_TIMESTAMP}.pt"

if best_fold.get("ckpt_path") and os.path.exists(best_fold["ckpt_path"]):
    shutil.copy(best_fold["ckpt_path"], best_ckpt_dst)
    print(f"Best checkpoint: {best_ckpt_dst}  (fold {best_fold['fold']}, AUC={best_fold['best_auc']:.4f})")

full_results = {
    "run_timestamp": RUN_TIMESTAMP,
    "config": cv_summary["config"],
    "dataset": {"n_proteins": len(proteins), "n_residues": int(total_res), "frac_disorder": float(frac_dis)},
    "pooled": {"auc": float(our_auc), "ap": float(our_ap), "f1": float(our_f1),
               "mcc": float(our_mcc), "opt_threshold": float(opt_thresh)},
    "per_fold": [{"fold": r["fold"], "best_auc": r["best_auc"], "best_ap": r["best_ap"],
                  "train_time_h": r["total_time"] / 3600, "history": r["history"]} for r in fold_results],
    "best_checkpoint": best_ckpt_dst,
    "biological_utility": bio_report,
    "af_rescue": af_report,
    "af3_rescue": af3_report,
    "af2_af3_comparison": af3_comparison,
    "af2_af3_breakthrough": breakthrough_summary,
    "caid_evaluation": caid_report,
    "statistical_validation": stat_validation,
    "structure_calibration": calibration_report,
    "phase3_integrated": phase3_report,
    "benchmark_tables": benchmark_report,
    "inference_fusion": fusion_report,
    "af3_combined_fusion": af3_fusion_report if "af3_fusion_report" in dir() else {"skipped": True},
    "gpu_v6_ensemble": ensemble_report,
}
results_path = f"disordernet_gpu_results_{RUN_TIMESTAMP}.json"
with open(results_path, "w") as f:
    json.dump(full_results, f, indent=2)

print(f"\n{'═'*60}")
print(" DISORDERNET GPU — FINAL RESULTS")
print(f"{'═'*60}")
print(f"  AUC-ROC  : {our_auc:.4f}")
print(f"  Avg Prec : {our_ap:.4f}")
print(f"  F1       : {our_f1:.4f}")
print(f"  Total CV : {total_cv_h:.2f} h")
print(f"  Saved    : {results_path}")
print(f"{'═'*60}")